# Modeling

## Data preparation

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path

base_dir = Path.cwd().parent
raw_data = base_dir / "data" / "raw"
processed_data = base_dir / "data" / "processed"

df = pd.read_csv(raw_data / "predictive_maintenance.csv")

features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

X = df[features]
y = df["Target"]


x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

ct = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Type']),         
        ('num', StandardScaler(), [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
    ])  
    ]
)

ct_rf = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Type']),
        ('num', 'passthrough', [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
    ])  
    ]
)

pipeline = Pipeline([
    ('preprocessing', ct),
    ('model', LogisticRegression(class_weight="balanced"))
])

pipeline_rf = Pipeline([
    ('preprocessing', ct_rf),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

## Model learning(Logistic Regression)

In [9]:
pipeline.fit(x_train, y_train)

print("Model learning completed successfully!")

Model learning completed successfully!


## Evaluation(Logistic Regression)
- the model “predicts failure” ~6x more often than it occurs
- the model detects most failures

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix

y_pred = pipeline.predict(x_test)

probabilities = pipeline.predict_proba(x_test)
probs = pipeline.predict_proba(x_test)[:, 1]

print("Probabilities of failure: ")
print(probabilities[:, 1])

precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Probabilities of failure: 
[0.97135024 0.5379081  0.68019628 ... 0.14890765 0.59522328 0.01710193]
Precision: 0.14
Recall: 0.82
F1 Score: 0.24

Confusion Matrix:
[[1593  339]
 [  12   56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000



## Tresholds comparision(Logistic Regression)
- Model can't separate classes well, it just shifts the decision boundary.
- Logistic Regression is too simple for this problem

In [16]:
thresholds = [0.5, 0.4, 0.3, 0.2]

for t in thresholds:
    y_pred = (probs >= t).astype(int)
    
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='binary')
    f1 = f1_score(y_test, y_pred, average='binary')
    
    print(t, precision, recall, f1)

0.5 0.14177215189873418 0.8235294117647058 0.24190064794816415
0.4 0.11509433962264151 0.8970588235294118 0.2040133779264214
0.3 0.09197080291970802 0.9264705882352942 0.16733067729083664
0.2 0.07246376811594203 0.9558823529411765 0.13471502590673576


## Model learning(Random Forest)

In [2]:
pipeline_rf.fit(x_train, y_train)

print("Model learning completed successfully!")

Model learning completed successfully!


## Evaluation(Random Forest)
- The model makes few false alarms.
- The model misses ~50% of failures.

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix

y_pred_rf = pipeline_rf.predict(x_test)
probs_rf = pipeline_rf.predict_proba(x_test)[:, 1]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Confusion Matrix:
[[1928    4]
 [  32   36]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.90      0.53      0.67        68

    accuracy                           0.98      2000
   macro avg       0.94      0.76      0.83      2000
weighted avg       0.98      0.98      0.98      2000



## Tresholds comparision(Random Forest)
- The model works best for a threshold of 0.3
- With a smaller threshold, recall increases, but unfortunately precision also decreases.

In [20]:
thresholds = [0.5, 0.4, 0.3, 0.2]

for t in thresholds:
    y_pred_rf = (probs_rf >= t).astype(int)
    
    precision = precision_score(y_test, y_pred_rf, average='binary')
    recall = recall_score(y_test, y_pred_rf, average='binary')
    f1 = f1_score(y_test, y_pred_rf, average='binary')
    
    print(t, precision, recall, f1)

0.5 0.9 0.5294117647058824 0.6666666666666666
0.4 0.8113207547169812 0.6323529411764706 0.7107438016528925
0.3 0.7222222222222222 0.7647058823529411 0.7428571428571429
0.2 0.5729166666666666 0.8088235294117647 0.6707317073170732


## Feature importance(Random Forest)
- The model makes decisions mainly based on torque and rotational speed
- Temperatures have an influence, but it is secondary
- Type does NOT add much information over and above other features

In [6]:
model = pipeline_rf.named_steps['model']
importances = model.feature_importances_

feature_names = pipeline_rf.named_steps['preprocessing'].get_feature_names_out()

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp.head(10))

                        feature  importance
6              num__Torque [Nm]    0.313735
5   num__Rotational speed [rpm]    0.245826
7          num__Tool wear [min]    0.169828
3      num__Air temperature [K]    0.127770
4  num__Process temperature [K]    0.121200
1                   cat__Type_L    0.010838
2                   cat__Type_M    0.006732
0                   cat__Type_H    0.004071


## Experimental feature selection(Random Forest)
- Type doesn't add any new information over and above other features.
- It's worth removing it for better generalization, less noise, and a simpler model.

In [4]:
ct_rf_without_type = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
    ])  
    ]
)

pipeline_rf_without_type = Pipeline([
    ('preprocessing', ct_rf_without_type),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline_rf_without_type.fit(x_train, y_train)

y_pred_rf_without_type = pipeline_rf_without_type.predict(x_test)
probs_rf_without_type = pipeline_rf_without_type.predict_proba(x_test)[:, 1]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_without_type))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_without_type))


Confusion Matrix:
[[1927    5]
 [  25   43]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.90      0.63      0.74        68

    accuracy                           0.98      2000
   macro avg       0.94      0.81      0.87      2000
weighted avg       0.98      0.98      0.98      2000

